## Aim:
How to link the rule-based extracted CI_TYPE-GEO pairs with the respective Ci failure impacts
 
**Idea**\
Test using prompt engineering by passing table of CI-GEO pairs to GPT-J model.
Steps:
* Load model and apply it always on one chunk of the document to extract CI failure impacts 
* Use prompt engineering to extract time and location of the CI failure (origin) the CI impacts (impact location)
or 
* Pass dataframe of pairs as input to the model
or
* Use few shot prompting with example answers

**Finally:**
* Compaire all approaches of spatial and temporal linking CI failure impacts


In [1]:
%load_ext scalene


Scalene extension successfully loaded. Note: Scalene currently only
supports CPU+GPU profiling inside Jupyter notebooks. For full Scalene
profiling, use the command line version. To profile in line mode, use
`%scrun [options] statement`. To profile in cell mode, use `%%scalene
[options]` followed by your code.


In [2]:
%%scalene --reduced-profile    # --cpu-only --cpu-sampling-rate 0.0001
# kernel crashes


Scalene: The specified code did not run for long enough to profile.
By default, Scalene only profiles code in the file executed and its subdirectories.
To track the time spent in all files, use the `--profile-all` option.


In [3]:

#
import os


# # settings for CUDA and PYTORCH
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
print(os.environ["CUDA_VISIBLE_DEVICES"])
# os.environ["CUDA_VISIBLE_DEVICES"]="0"
os.environ["PYTORCH_ALLOC_CONF"]="expandable_segments:True" ## improve memory allocation

# # settings for debugging CUDA errors (pinpoint exact line of error)
os.environ["TORCH_USE_CUDA_DSA"] = "1"
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1" 


# ### settings for distributed computing
# import torch
# from torch import distributed as dist
# from transformers import AutoTokenizer, AutoModelForCausalLM

# # NOTE find MASTER_ADDR and MASTER_PORT  (any unused port)
# # hostname -I | awk '{print $1}'
# # comm -23 <(seq 29500 30000 | sort) <(ss -Htan | awk '{print $4}' | cut -d: -f2 | sort -u) | head -n 1
# os.environ["WORLD_SIZE"]="1"
# os.environ["RANK"]="0"
# os.environ["LOCAL_RANK"]="0"
# os.environ["MASTER_ADDR"]="10.10.0.2" # "localhost"
# os.environ["MASTER_PORT"]="29500" #"12355" #

# local_rank = int(os.environ["LOCAL_RANK"])
# torch.cuda.set_device(local_rank)
# dist.init_process_group(backend="nccl")


# activate global venv explicitly
os.environ["VIRTUAL_ENV"] = "/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/.venv"


import torch

print(torch.cuda.is_available())
print(torch.cuda.device_count())  # should give 2
print(torch.cuda.get_device_name())
print(torch.cuda.get_device_properties(0))
# print(torch.cuda.get_device_properties(1))
print(torch.cuda.get_device_capability())
print(torch.cuda.get_arch_list())
print(torch.__version__)
print(torch.version.cuda)


## --> must be CUDA 12.6, torch: 2.91, ['sm_50', 'sm_60', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90']

import os
import sys
# import subprocess
import re
import time
from glob import glob
from pathlib import Path
import gc
from typing import List, Dict, Tuple, Optional, Union
# from io import StringIO
import json

# from tqdm import tqdm
import numpy as np
import pandas as pd
from fuzzywuzzy import fuzz
import geonamescache
# import pyarrow as pa
# import pyarrow.parquet as pq
import spacy
from huggingface_hub import login
from pdfminer.high_level import extract_text
import langdetect
from transformers import AutoTokenizer
from haystack.dataclasses import ByteStream
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from docling_core.types import DoclingDocument
from docling_core.types.doc import DocItemLabel
from docling_core.types.doc.document import SectionHeaderItem, ListItem, TextItem, DocItem
from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType
# from langchain.document_loaders import DirectoryLoader #, UnstructuredLoader
# from langchain_community.document_loaders import DirectoryLoader, UnstructuredLoader
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    AcceleratorOptions,
    AcceleratorDevice,
)
from docling.pipeline.standard_pdf_pipeline import StandardPdfPipeline
from docling.document_converter import DocumentConverter, FormatOption
from docling.chunking import HybridChunker



from src.settings import settings as s
import src.document_cleaning as dc
import src.translation_model as tm
import src.extraction_model as em
import src.postprocess as pp
import src.utils as u
from geollama.geollama.main import GeoLlama
from geollama.geollama.model import TopoModel, RAGModel



test_mode = True

# login to HF
# login(token=os.getenv("HUGGINGFACE_TOKEN"))
try: 
    login(token=os.getenv("HUGGINGFACE_TOKEN"))   # notebook_login
except:
    login(token=os.environ.get("HUGGINGFACE_TOKEN"))  # former HF_TOKEN
        
# NOTE raises exception if not env.variable doesnt exist (compared to os.envrion.get and its shortcut os.getenv)


# NOTE. disabled batch size as OOM for CUDA despite chunkwise memory cleaning, nvtop to find best batchsize
BATCH_SIZE = s.BATCH_SIZE  # max for nvidia GPU

# torch.manual_seed(42)

#  automatic linebreaks and multi-line cells.
pd.set_option('display.max_colwidth', 100000)
pd.set_option("display.colheader_justify", "left")

print(os.environ["CUDA_VISIBLE_DEVICES"])

# clean up before applying CUDA
# gc.collect()
# torch.cuda.empty_cache() 
print(torch.cuda.memory_reserved() / 1e9)
torch.no_grad()



# ## TODO test to prevent CUDA-OOM when reused
# ## Source: https://spacy.io/usage/embeddings-transformers
# from thinc.api import set_gpu_allocator, require_gpu

# # Use the GPU, with memory allocations directed via PyTorch.
# # This prevents out-of-memory errors that would otherwise occur from competing
# # memory pools.
# set_gpu_allocator("pytorch")
## require_gpu(0)



# %% [markdown]
# ## Set paths and vars

# %%
# set wd to project root
# os.chdir("/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval")

## set path variables
DOCS_DIR = Path(s.PATH_DATA / "text_sources/")
PARSED_TEXT_DIR = Path(s.PATH_DATA / "parsed_documents/")
NER_PATTERNS_FILEPATH = Path(s.NER_PATTERNS_FILEPATH)
LLM_OUTPUTS_DIR = Path(s.PATH_DATA / "llm_outputs/")
geollama_OUTPUTS_DIR = Path(s.PATH_DATA / "geollama_outputs/")

os.makedirs(PARSED_TEXT_DIR, exist_ok=True)
os.makedirs(s.PATH_LLM_DATA, exist_ok=True)
os.makedirs(geollama_OUTPUTS_DIR, exist_ok=True)


# CI GEO pairs
CI_GEO_FILEPATH = Path( s.PATH_DATA / s.CI_GEO_PAIRS_FILENAME)

## store LLM 1 response and prompt
OUTPUT_LLM1_FILEPATH =  Path(s.PATH_LLM_DATA / s.LLM_DATA_FILENAME)
OUTPUT_geollama_FILEPATH =  Path(s.PATH_LLM_DATA / "geollama_results.csv")




# %% [markdown]
# ### Set test mode

# %%

docs_list_sample = [

#         Path(PARSED_TEXT_DIR, "Chamra 2006 - Flooding of the Prague metro during the August 2002 floods.md"),
#         # Path(PARSED_TEXT_DIR, "Fink 2009 - The European storm Kyrill in January 2007_ synoptic evolution, meteorological impacts and some considerations with respect to climate change.md"),
#         # Path(PARSED_TEXT_DIR, "Hladny 2004 - August_2002_catastrophic_flood_in_the_Czech_Republic.md"),
#         # Path(PARSED_TEXT_DIR, "Mitsakis 2014 - Impacts of high-intensity storms on urban transportation_ applying traffic flow control methodologies for quantifying the effects.md"),
#         # Path(PARSED_TEXT_DIR, "Pescaroli 2017 - How Critical Infrastructure Orients International Relief in Cascading.md"),
    
#         Path(PARSED_TEXT_DIR, "Lloyd's List 2024 - Port of Valencia reopens after devastating floods.md"),
#         Path(PARSED_TEXT_DIR, "Containerlift 2024 - Valencia Port Resumes Operations Following Devastating Flooding in Spain - Containerlift.co.uk - Transport_Lifting_Shipping.md"), 
#         Path(PARSED_TEXT_DIR, "ABC 2024 - Traffic jams and flight delays due to heavy rain and lightning storm in Malaga.md"),

#         Path(PARSED_TEXT_DIR, "Karakatsani 2023 - Greece economy briefing The economic impact of the recent devastating floods in Greece.md"),
#         Path(PARSED_TEXT_DIR, "European Investment Bank 2025 - Spain_ EIB lends €50 million to Iberdrola to rebuild and climate-proof flood-hit power infrastructure in Valencia.md"),
#         Path(PARSED_TEXT_DIR, "Wilson 2024 - Flash floods in Spain sweep away cars, disrupt trains and leave several missing _ AP News.md"),     
#             Path(PARSED_TEXT_DIR, "Wildhagen 2013 - Hochwasser_ Wie die Flut Unternehmen lahmlegt.md"),
#             Path(PARSED_TEXT_DIR, "AFP 2022 - The_Vibes_Valencia Airport in Madrid briefly shut as lightning hits runway _ World _ The Vibes.md"),
#             Path(PARSED_TEXT_DIR, "Artemis 2015 - PERILS finalises Storm Desmond UK flood loss estimate at £604m.md"),
#             Path(PARSED_TEXT_DIR, "Brown 2010 - Economy feels chill as UK grinds to a halt _ The Independent.md"),
#         #     # Path(PARSED_TEXT_DIR, "Diakakis 2020 - A systematic assessment of the effects of extreme flash floods on transportation infrastructure and circulation: The example of the 2017 Mandra flood.md"),
#         Path(PARSED_TEXT_DIR, "EFE 2024 - The DANA storm, live_ The death toll rises to 158.md"),
#         #     # Path(PARSED_TEXT_DIR, "Eurelectric 2006 - Impacts of Severe Storms on Electric Grids.md"),
# Path(PARSED_TEXT_DIR, "Euronews 2024 - Spain floods_ Death toll rises to 205 as nation braces for more rain .md"),
# Path(PARSED_TEXT_DIR, "Ferlita 2023 - Incendi in Sicilia, ecco cosa accade.md"),
#           Path(PARSED_TEXT_DIR, "Fink 2004 - The 2003 European summer heatwaves and drought - synoptic diagnosis and impacts.md"),
#           Path(PARSED_TEXT_DIR, "Gilbody Dickerson 2024 - Spain floods_ At least 95 people killed including British man near Malaga _ World News _ Sky News.md"),
#           #     # Path(PARSED_TEXT_DIR, "Kadir 2014 - The Impact of Natural Disasters on Critical Infrastructures - A Domino Effect-based Study.md"),
#           Path(PARSED_TEXT_DIR, "Kaur 2025 - Authorities suspect arson in 17 wildfires across Dalmatian coast, Croatia - The Watchers.md"),
#           #     # Path(PARSED_TEXT_DIR, "Keller 2014 - Mapping Natural Hazard Impacts on Road Infrastructure—The Extreme Precipitation in Baden-Württemberg.md"),
        #     # Path(PARSED_TEXT_DIR, "Koks 2019 - Understanding Business Disruption and Economic Losses Due to Electricity Failures and Flooding.md"),
        #     # Path(PARSED_TEXT_DIR, "Korzilius 2021 Nach der Flut.md"),
# Path(PARSED_TEXT_DIR, "Kettle 2020 - Storm Xaver over Europe in December 2013 Overview of energy impacts and North Sea events.md"),

# Path(PARSED_TEXT_DIR, "Khazai 2013 - Juni-Hochwasser 2013 in Mitteleuropa - Fokus Deutschland Bericht 2 Auswirkungen und Bewältigung.md"),
# Path(PARSED_TEXT_DIR, "Rozendaal 2021 - Infrabel_ Flood damage to railway track worth tens of millions of euros _ SpoorPro - incomplete.md"),
# Path(PARSED_TEXT_DIR, "Skoulding 2023 - Where are the fires in Italy today as temperatures rise to 47.6C on Sicily_ _ The Independent.md"),
   #  Path(PARSED_TEXT_DIR, "Koks 2022 - Brief communication.md"),
    Path(PARSED_TEXT_DIR, "Treanor 2015 - Storm Desmond damage across Cumbria estimated at £500m _ Storm Desmond _The Guardian.md"),
    Path(PARSED_TEXT_DIR, "Nour 2011 - Damages caused by floods and flash-floods upon critical infrastructure.md"),
    
    # # # long processing
    Path(PARSED_TEXT_DIR, "AEMET 2024 - ESTUDIO SOBRE LA SITUACIÓN DE LLUVIAS INTENSAS.md"),
    Path(PARSED_TEXT_DIR, "Koks 2022 - Brief communication.md"),
    
    Path(PARSED_TEXT_DIR, "Fink 2009 - The European storm Kyrill in January 2007_ synoptic evolution, meteorological impacts and some considerations with respect to climate change.md"),
    Path(PARSED_TEXT_DIR, "Hladny 2004 - August_2002_catastrophic_flood_in_the_Czech_Republic.md"),
    Path(PARSED_TEXT_DIR, "Mitsakis 2014 - Impacts of high-intensity storms on urban transportation_ applying traffic flow control methodologies for quantifying the effects.md"),
    Path(PARSED_TEXT_DIR, "Pescaroli 2017 - How Critical Infrastructure Orients International Relief in Cascading.md"),
    # Path(PARSED_TEXT_DIR, "Nieuwsblad 2021 - Geen drinkbaar water, geen gas, en geen elektriciteit_ 9.000 mensen moeten op zoek naar ander onderkomen.md"),
    
    
    #     #     # not part of valid set:
    #     #     # Path(PARSED_TEXT_DIR, "Krausmann 2014 - STREST report on lessons learned from recent catastrophic events.md"), # > 1800 entries LLMv3.0 incl. hallucinations
]




## Test mode
if test_mode:
    search_path = docs_list_sample
    print("Test mode is ON. Using only a small sample of documents for testing.")
else:
    search_path = glob(str(Path(PARSED_TEXT_DIR, "*cleaned.jsonl")))

print(f"Using {len(search_path)} documents for processing.")




0
True
1
NVIDIA A100-PCIE-40GB
_CudaDeviceProperties(name='NVIDIA A100-PCIE-40GB', major=8, minor=0, total_memory=40441MB, multi_processor_count=108, uuid=f5986d5b-be6f-0c70-66b9-cc911c9fbfe9, pci_bus_id=130, pci_device_id=0, pci_domain_id=0, L2_cache_size=40MB)
(8, 0)
['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
2.8.0+cu128
12.8


/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/.venv/lib/python3.12/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


Running on TUB cluster
Running on TUB Cluster


0
0.0
Test mode is ON. Using only a small sample of documents for testing.
Using 8 documents for processing.


In [4]:
print(os.environ["CUDA_VISIBLE_DEVICES"])
#os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
print(os.environ["CUDA_VISIBLE_DEVICES"])


0
0


In [5]:
# # # # settings for CUDA and PYTORCH
# # os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
# # print(os.environ["CUDA_VISIBLE_DEVICES"])
# # # os.environ["CUDA_VISIBLE_DEVICES"]="0"
# # os.environ["PYTORCH_ALLOC_CONF"]="expandable_segments:True" ## improve memory allocation

# # # # settings for debugging CUDA errors (pinpoint exact line of error)
# # os.environ["TORCH_USE_CUDA_DSA"] = "1"
# # # os.environ["CUDA_LAUNCH_BLOCKING"] = "1" 

# # # activate global venv explicitly
# # os.environ["VIRTUAL_ENV"] = "/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/.venv"

# import os 
# import torch

# print(torch.cuda.is_available())
# print(torch.cuda.device_count())  # should give 2
# print(torch.cuda.get_device_name())
# print(torch.cuda.get_device_properties(0))
# print(torch.cuda.get_device_properties(1))
# print(torch.cuda.get_device_capability())
# print(torch.cuda.get_arch_list())
# print(torch.__version__)
# print(torch.version.cuda)
# # has two gpus nabled? 


# # # # settings for CUDA and PYTORCH
# # # os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
# # print(os.environ["CUDA_VISIBLE_DEVICES"])
# # os.environ["TORCH_USE_CUDA_DSA"] = "1"

# # # os.environ["CUDA_VISIBLE_DEVICES"]="0"
# # # # os.environ["PYTORCH_ALLOC_CONF"]="expandable_segments:True" ## improve memory allocation

# # # # # settings for debugging CUDA errors (pinpoint exact line of error)
# # # os.environ["TORCH_USE_CUDA_DSA"] = "1"
# # # # os.environ["CUDA_LAUNCH_BLOCKING"] = "1" 

# print(os.environ["CUDA_VISIBLE_DEVICES"])

# # # settings for CUDA and PYTORCH
# os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
# print(os.environ["CUDA_VISIBLE_DEVICES"])
# os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
# os.environ["PYTORCH_ALLOC_CONF"]="expandable_segments:True" ## improve memory allocation

# # # settings for debugging CUDA errors (pinpoint exact line of error)
# os.environ["TORCH_USE_CUDA_DSA"] = "1"
# print(os.environ["CUDA_VISIBLE_DEVICES"])

# print(torch.cuda.is_available())
# print(torch.cuda.device_count())
# # %%
# # Settings
# #model_name = "meta-llama/Llama-3.1-8B-Instruct"
# model_name = "meta-llama/Meta-Llama-3-70B-Instruct"


# from huggingface_hub import login
# from transformers import (
#     AutoTokenizer,
#     AutoModelForCausalLM,
#     BitsAndBytesConfig,
#     DynamicCache,
# )

# from src.settings import settings as s
# import src.utils as u


# model_dir = s.HF_HOME_DIR   # use default dir in .cache/
        
# print(f"Model directory: {model_dir}")

# # quantization config
# # Load model with 4-bit quantization if applicable (use 4-bit integer instead of 32b floats) --> reduce the required VRAM for model application
# # see, https://huggingface.co/docs/transformers/quantization
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
# )
# flash_attn_config = "flash_attention_2"

# model = AutoModelForCausalLM.from_pretrained(
#         model_name,
#         cache_dir=model_dir,
#         local_files_only=True,  # tp_plan="auto" # set tensor parallel model (ie. splits model on multiple GPU)
#         # max_seq_length=2048,
#         dtype="auto", # None ,# test for CU12.6, torch.29.1 #"auto",
#         device_map="auto",
#         attn_implementation=flash_attn_config,
#         quantization_config=bnb_config,
#         # tp_plan="auto",  # automatically use a tensor parallelism plan based on predefined configuration of the model (i.e. partition model on both GPUs)
#     )
# print(model.hf_device_map)
# {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 0, 'model.layers.13': 0, 'model.layers.14': 0, 'model.layers.15': 0, 'model.layers.16': 0, 'model.layers.17': 0, 'model.layers.18': 0, 'model.layers.19': 0, 'model.layers.20': 0, 'model.layers.21': 0, 'model.layers.22': 0, 'model.layers.23': 0, 'model.layers.24': 0, 'model.layers.25': 0, 'model.layers.26': 0, 'model.layers.27': 0, 'model.layers.28': 0, 'model.layers.29': 0, 'model.layers.30': 0, 'model.layers.31': 1, 'model.layers.32': 1, 'model.layers.33': 1, 'model.layers.34': 1, 'model.layers.35': 1, 'model.layers.36': 1, 'model.layers.37': 1, 'model.layers.38': 1, 'model.layers.39': 1, 'model.layers.40': 1, 'model.layers.41': 1, 'model.layers.42': 1, 'model.layers.43': 1, 'model.layers.44': 1, 'model.layers.45': 1, 'model.layers.46': 1, 'model.layers.47': 1, 'model.layers.48': 1, 'model.layers.49': 1, 'model.layers.50': 1, 'model.layers.51': 1, 'model.layers.52': 1, 'model.layers.53': 1, 'model.layers.54': 1, 'model.layers.55': 1, 'model.layers.56': 1, 'model.layers.57': 1, 'model.layers.58': 1, 'model.layers.59': 1, 'model.layers.60': 1, 'model.layers.61': 1, 'model.layers.62': 1, 'model.layers.63': 1, 'model.layers.64': 1, 'model.layers.65': 1, 'model.layers.66': 1, 'model.layers.67': 1, 'model.layers.68': 1, 'model.layers.69': 1, 'model.layers.70': 1, 'model.layers.71': 1, 'model.layers.72': 1, 'model.layers.73': 1, 'model.layers.74': 1, 'model.layers.75': 1, 'model.layers.76': 1, 'model.layers.77': 1, 'model.layers.78': 1, 'model.layers.79': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}


## Document cleaning

In [6]:
# # load tokenizer
# embed_model =  "sentence-transformers/all-MiniLM-L6-v2"
# tokenizer = HuggingFaceTokenizer(
#     tokenizer=AutoTokenizer.from_pretrained(embed_model),
#     max_tokens=256, # max tokens for MiniLM-l6-v2, set here explicitly
#     # standardize input sizes of chunks for Llama models
#     padding=True, # add zero as extra tokens to too short sequences so that they have the same length as other chunks
#     truncation=True, # truncates too long sequences (> max_tokens). If False, they will be split into multiple chunks
# )

# ## init chunker - based on hierachical chunker but also considers max token leng, merge smaller chunks, except when at end of paragraph (merge_peers=True)
# chunker = HybridChunker(
#     tokenizer=tokenizer,
#         # max_tokens=256, # max tokens for MiniLM-l6-v2, set here explicitly
#         # chunk_overlap=0, # no overlap between chunks, as we use merge_peers to merge smaller chunks and avoid splits in sentences
#     split_by_sentence=True, # split by sentence first before merging smaller chunks, to avoid splits in sentence middle
#     merge_peers=True,  # optional, defaults to True
# )



In [7]:

print( "Number of documents to process:", len(os.listdir(DOCS_DIR)) )
# print("Number of documents to process:", len(search_path) )
start_time = time.time()


# for Doc parsing and cleaning
md_converter = DocumentConverter(allowed_formats=[InputFormat.MD])



## Start CI impact extraction
for file_no, filename in enumerate(search_path):

    EXPORT_TYPE = ExportType.DOC_CHUNKS
    mislabeled: List[DocItem] = []
    min_paragraph_size = 50  
    # temp_docs: List[ByteStream] = []
    temp_meta: List[Dict[str, str]] = []
    i: int
    combined_paragraph: str = ""
    combined_chars: int = 0
    para_num: int = 0
    section_name: str = ""
    page_no: Optional[int] = None
    first_note: bool = False


    time1 = time.time()

    src_language_nonengl = None 

    no_documents = len(search_path)
    filepath = Path(filename)
    filename_stem = filepath.stem


    print(f"\n\n ######## -------- Processing document [{file_no+1}/{no_documents}]: {filepath.name} -------- ######## \n")

    ## extract authors, publication year and title 
    author, year, title = dc.extract_citation_info(filename_stem)
    citation = f"{author} {year}".replace("  ", " ").strip()
    title = title.replace(" - ", "").replace("_cleaned", "").strip()


  
    print(f"\n ##### ------- Cleaning document -----------########")

    pdf_filepath = os.path.join("../", DOCS_DIR, Path(filename_stem + ".pdf"))
    md_filename = filename_stem + ".md"
    md_filepath = os.path.join(PARSED_TEXT_DIR, Path(md_filename))
    cleaned_md_filepath = md_filepath.replace(".md", "_cleaned.md")
    # cleaned_jsonl_filepath = md_filepath.replace(".md", "_cleaned.jsonl")

    time_cleaning = time.time()

    if os.path.exists(cleaned_md_filepath):
        print(f"Cleaned markdown file already exists, loading file and proceeding with Ci impact extraction")
        # Load the existing cleaned markdown file
        doclingdoc = md_converter.convert(cleaned_md_filepath).document      

    else:
        print(f"Cleaned markdown file does not exist yet. Cleaning document: '{filename}'")


        # get language of document
        src_language_doc = langdetect.detect(filename_stem.lower())  # lower case improves language detection


        ## Document converter with OCR
        print("Using OCR for text extraction as it identifies section titles, footers/headers and pagenumbers as such, but reads in also figure text sometimes") 
        # NOTE all other standard doclingConverter retunr section/headers etc as BODY not FURNITURE
        # NOTE: partly reads in figure text and table text 
        pdf_doc_org = dc.DocumentParser().ocr_converter.convert(source=pdf_filepath).document  ## !! recognizes section titles, footers/headers !! :D

        ##  get only list of Doc.items
        texts = dc.get_processed_texts(pdf_doc_org) 

        texts_clean = []
        section_names = []
        for i, text in enumerate(texts):
            

 ### as Parser Doc class
            # get next text only when it is not page header/footer
            next_text = dc.get_next_text(texts, i)
            # page_no = get_current_page(text, combined_paragraph, page_no)


            # Update section header if the element is a section header
            # TODO: Need a stronger check on section headers that takes top of page into account, etc
            if dc.is_section_header(text) and text not in mislabeled:
                print("!!  Section header found:", text.text)
                section_name = text.text
                continue

            if dc.is_reference_section(section_name):
                print("Reference section found. Stopping further processing of document.")
                break  

            if dc.should_skip_element(text):
                continue
            
            # clean from double whitespace, newlines, etc.
            p_str = dc.clean_text(text.text)

            # clean from potential figure references
            p_str = dc.remove_figure_references(p_str)

            ## replace e.g. and i.e. --> eg and ie to avoid sentence splits
            p_str = re.sub(r"e\.g\.\s+", "eg ", p_str)
            p_str = re.sub(r"i\.e\.\s+", "ie ", p_str)

            # Removing URLs 
            # LangExtract tries to open these URLs when they occur in the document text
            # p_str= re.sub(r"http\S+", "", p_str) 

            p_str_chars = len(p_str)

            # If the paragraph does not end with final punctuation, accumulate it
            if not dc.is_sentence_end(p_str):
                combined_paragraph = dc.combine_paragraphs(combined_paragraph, p_str)
                combined_chars += p_str_chars
                continue

            # p_str ends with a sentence end; decide whether to process or accumulate it
            total_chars = combined_chars + p_str_chars
            if dc.is_section_header(next_text):
                # Immediately process if the next text is a section header
                p_str = dc.combine_paragraphs(combined_paragraph, p_str)
                combined_paragraph, combined_chars = "", 0
            elif total_chars < min_paragraph_size:
                # Not enough characters accumulated yet; decide based on next_text
                if next_text is None or (not dc.is_page_text(next_text) and dc.is_sentence_end(p_str)):
                    # End of document or next text item is not a text item and current paragraph ends with punctuation
                    # Process the paragraph and reset the accumulator even though this is a short paragraph
                    p_str = dc.combine_paragraphs(combined_paragraph, p_str)
                    combined_paragraph, combined_chars = "", 0
                else:
                    # Combine with next paragraph
                    combined_paragraph = dc.combine_paragraphs(combined_paragraph, p_str)
                    combined_chars = total_chars
                    continue
            else:
                # Sufficient characters: process the paragraph and reset the accumulator
                p_str = dc.combine_paragraphs(combined_paragraph, p_str)
                combined_paragraph, combined_chars = "", 0

            p_str = dc.combine_hyphenated_words(p_str)
            if p_str:  # Only add non-empty content
                para_num += 1
                dc.add_paragraph(
                    p_str, 
                    #para_num, section_name, page_no, 
                    temp_docs, 
                    # temp_meta
                )
                page_no = None
            
            print("\n-__ Paragraph #", para_num, "; Section:", section_name)

            # print(p_str)
            # print(temp_docs, temp_meta)

##### as CLASS DCOPAraser=
### translator class
            ## Translation
            if src_language_doc != "en":

                supported_languages = ["fr", "de", "es", "it", "nl"]
                # supported_languages = ["en", "fr", "de", "es", "pt", "it", "pl", "cs", "nl", "da", "sv", "no", "hr", "ro", "bg", "sl", "sk", "lt", "et" ],
                if src_language_doc not in supported_languages:
                    print(f"Unsupported source language: {src_language_doc}. Continue with extraction on original text")
                    continue 

                print(f"Translating {src_language_doc} --> en")
                p_str = tm.translate_2_english(src_language_doc, p_str)     
                
            # collect cleaned text + meta data per doc            
            texts_clean.append(p_str)
            section_names.append(section_name)


        print("TEST Saving cleaned text as DoclingDocument to store text (p_str)+ meta (sectiontitle, page_no): ")
        # Initialize new DoclingDoc
        doclingdoc = DoclingDocument(schema_name="DoclingDocument",  version="1.0.0",name="My Custom Document")
        
        # write text and structural elements (titles, sections, and paragraphs) to doc     
        # title_node = doclingdoc.add_title(text=f"{filename.replace('.pdf', '')}")
        title_node = doclingdoc.add_title(text=filename_stem)
        
        for text, section_name in zip(texts_clean, section_names):
            print(section_name, ":", text)
            section_node = doclingdoc.add_heading(text=section_name, level=1, parent=title_node)
            doclingdoc.add_text(label=DocItemLabel.TEXT, text=text, parent=section_node)

        # Save doclingDocument as MD
        doclingdoc.save_as_markdown(cleaned_md_filepath)     


       ## STEP 2: then do chunking with HybridChunker  TODO
        # # NOTE maybe only a bit needed as already very good splits but maybe tokensizes need to be adapted 
        ## TODO inlcude p_str always as doclingobj part or write it back to DoclingObject, (maybe with section_name as meta info)

        # chunking
        chunk_iter = chunker.chunk(dl_doc=doclingdoc)
        chunks = list(chunk_iter)

        # # apply contextualization
        # ser_text = chunker.contextualize(chunk=chunk)
        # ser_tokens = tokenizer.count_tokens(ser_text)
        # print(f"chunker.contextualize(chunk) ({ser_tokens} tokens):\n{ser_text!r}")
        # print()

        break


end_time = time.time() - start_time
print(f"Parsing and cleaning done. Time elapsed: {end_time:.2f} seconds.")


# visual check of removed items
# TODO make as document_cleaning function: print removed items with largest number of chars first
# ## NOTE. high number of chars == more potentially actual text body

# text_items_removed = sorted(text_items_to_drop_visualization, key=lambda x: -x[0])
# for i in text_items_removed[:50]:
#     print(i) # -->  also subsection titles were removed partly



Number of documents to process: 67


 ######## -------- Processing document [1/8]: Treanor 2015 - Storm Desmond damage across Cumbria estimated at £500m _ Storm Desmond _The Guardian.md -------- ######## 


 ##### ------- Cleaning document -----------########
Cleaned markdown file does not exist yet. Cleaning document: '/beegfs/scratch/a-buch/_PROJECTS/data/parsed_documents/Treanor 2015 - Storm Desmond damage across Cumbria estimated at £500m _ Storm Desmond _The Guardian.md'
Using OCR for text extraction as it identifies section titles, footers/headers and pagenumbers as such, but reads in also figure text sometimes


ImportError: EasyOCR is not installed. Please install it via `pip install easyocr` to use this OCR engine. Alternatively, Docling has support for other OCR engines. See the documentation.

In [10]:
from pathlib import Path
import re
import warnings
from contextlib import suppress

import json
from typing import Iterable
from docling_core.types.doc.document import TextItem
from langchain_core.documents import Document
from docling.datamodel.pipeline_options import PdfPipelineOptions, EasyOcrOptions, AcceleratorOptions
from docling.document_converter import ConversionResult, DocumentConverter, PdfFormatOption, PipelineOptions, InputFormat




class DocumentParser:
    """Class for parsing and cleaning documents."""

    def __init__(self):
                
        # OCR pipeline configs
        self.artifacts_path = Path("../docling_artifacts")
        self.artifacts_path.mkdir(exist_ok=True)

        # OCR and pipeline options
        ocr_options = EasyOcrOptions(
            lang = ["fr", "de", "es", "it", "nl"],
            #lang=["en", "fr", "de", "es", "pt", "it", "pl", "cs", "nl", "da", "sv", "no", "hr", "ro", "bg", "sl", "sk", "lt", "et" ],
            download_enabled=True
        )
        pipeline_opts = PdfPipelineOptions(
            artifacts_path=self.artifacts_path,
            do_ocr=True,         # Required for text extraction
            do_table_structure=False,  # Disable table analysis if not needed
            # allow_external_plugins=True,
            ocr_options=ocr_options
        )
        # PDF format options
        pdf_format_option = PdfFormatOption(
            pipeline_options=pipeline_opts,
            reading_order="natural"
        )
        # init pdf converter
        self.ocr_converter = DocumentConverter(
            format_options={InputFormat.PDF: pdf_format_option}
        )

pdf_doc_org = DocumentParser().ocr_converter.convert(source=pdf_filepath).document 

ImportError: EasyOCR is not installed. Please install it via `pip install easyocr` to use this OCR engine. Alternatively, Docling has support for other OCR engines. See the documentation.

In [12]:
!uv add easyocr

Resolved 316 packages in 1.56s                                       
Prepared 1 package in 4.66s                                              ⠋ Preparing packages... (0/0)                                                   
Uninstalled 1 package in 10ms
Installed 7 packages in 583ms                               
 ~ ci-impacts-llm==0.1.0 (from file:///beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval)
 + easyocr==1.7.2
 + imageio==2.37.3
 + lazy-loader==0.5
 + python-bidi==0.6.10
 + scikit-image==0.26.0
 + tifffile==2026.6.1


#### Llama 4 or Llama-3-70 b test

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""Data mining model (decoder, tokenizer) for extracting impacts on infrastructure"""

__author__ = "Anna Buch, TU Berlin"
__email__ = "anna.buch@tu-berlin.de"



import os
import copy
import json 
from typing import Optional, List
from pathlib import Path
from jinja2 import Environment, FileSystemLoader

import pandas as pd
from openai import OpenAI
from pydantic import BaseModel

from huggingface_hub import login
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DynamicCache,
)
from transformers.distributed import DistributedConfig
from openai import OpenAI

import torch

from src.settings import settings as s
import src.utils as u
from src import postprocess as pp
        



class ResponseItems(BaseModel):
    infrastructure_type: str
    damage: Optional[str] = None
    damage_value: Optional[str | list[str] | int | float] = None 
    location: str
    location_type: Optional[str] = None

class Response(BaseModel):
    impact_cases: List[ResponseItems]
    


def load_prompt_template(
    template_path: str = "./prompt_templates",
    template_filename: str = None,
):
    env = Environment(loader=FileSystemLoader(template_path))
    template = env.get_template(template_filename)

    return template

                

class OpenAIModel:

    def __init__(self, model_name, system_prompt: load_prompt_template):
        
        # try: 
        #     login(token=os.getenv("HUGGINGFACE_TOKEN"))   # notebook_login
        # except:
        #     login(token=os.environ.get("HUGGINGFACE_TOKEN"))  # former HF_TOKEN
        # s.HF_HOME_DIR = "/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/notebooks/huggingface_mirror/hub"
        # base_dir = s.HF_HOME_DIR   # use default dir in .cache/
        # model_dir = Path(base_dir)  # / f"models--{model_name.replace("/", "--")}"  # is already .._mirror/hub/
        
        self.system_prompt = system_prompt
        self.model_name = model_name
        self.client = OpenAI(
            base_url = "https://integrate.api.nvidia.com/v1",
            api_key="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"
        )

        print(f"Model calling via nvidia API:")

    #     self.client = self.initialize_model(
    #         client,
    #         system_prompt=system_prompt,
    #     )
        

    # def initialize_model(
    #     self, client: OpenAI, 
    #     system_prompt=None, 
    #     max_new_tokens: int = 2048 # 4096
    # ):
            
    #     # use flash-attn when GPU type supports it (e.g., A100, not support:tesla P100)
    #     flash_attn_config = None
    #     if u.supports_flash_attention(0):  # check only for first GPU
    #         print("Using flash attention")
    #         flash_attn_config = "flash_attention_2"
        
    #     system_prompt = system_prompt.render()



    def generate_response(
        self, 
        # question: str, 
        # context: list, 
        user_prompt: str, #load_prompt_template, 
        # user_dynamic_prompt: load_prompt_template, 
        top_k: int = None,
        top_p: float = None,
        temperature: float = 0.01,
        max_new_tokens: int = 1024
    ):
        system_prompt = self.system_prompt.render()
        # user_prompt = user_prompt.render()
        # user_dynamic_prompt = user_dynamic_prompt.render(
        #     context=context,  # includes also df_ci_geo info
        #     question=question,
        # )
        response = self.client.chat.completions.parse(
        # with client.chat.completions.stream(
        # response = client.responses.parse(  # use it with Pydantic List[str] for multi-output
            model=self.model_name, # "openai/gpt-oss-20b", #:fireworks-ai",
            temperature=0.0,
            top_p=0.01,
            reasoning_effort= "low",
            # seed=42,
            #stream=False,
            # max_completion_tokens=1024,
            #max_tokens=1024,
            # prompt_cache_key
            # prompt_cache_retention= "24h",  # test past_key_value (iterative caching)
            messages=[
                {"role": "system", "content": f"{self.system_prompt}"},
                #{"role": "developer", "content": f"These are the single steps you should conduct for extracting the information about the impacts to critical infrastructure assets: {user_content}",},
                {"role": "user", "content": f"""{user_prompt}"""},
            ],
            #stop=["<extraction>"],
            response_format=Response,
            #include=["infrastructure_type", "location"],
        )
        # ) as stream:
        #     for event in stream:
        #         if event.type == "response.refusal.delta":
        #             print(event.delta, end="")
        #         elif event.type == "response.output_text.delta":
        #             print(event.delta, end="")
        #         elif event.type == "response.error":
        #             print(event.error, end="")
        #         elif event.type == "response.completed":
        #             print("Completed") # print(event.response.output)
        
        #     # final_response = stream.get_final_response()
        #     # print(final_response)

        try:
            parsed_output = json.loads(response.to_json())["choices"][0]["message"]["content"]

            return parsed_output

        except Exception as e:
            print("Could not parse response as JSON:", e)
            parsed_output = None
            for output in response.choices[0]:
                if output[0] != "message":
                    continue
                for item in output[1].content:
                    if item.type == "refusal":
                        # If the model refuses to respond, you will get a refusal message
                        print(item.refusal)
                        continue
                    if not item.parsed:
                        raise Exception("Could not parse response")
                    print(item.parsed)


In [ ]:
# client.responses.parse?

In [ ]:
# text2 = "There have also been some traffic jams and road closures in the capital, such as on Lope de Vega Avenue, heading towards the city center, near Julio Cortázar Avenue due to a road subsidence. As the City Council indicated on social media, traffic has been diverted towards Colonia Santa Inés. There is also heavy traffic on the access roads from Juan XXIII Avenue to Plaza Manuel Azaña, Blas Infante Avenue, Guerrero Strachan Avenue, Velázquez Avenue, Camino Suárez, the Azucarera - Interhorce road, Santa Rosa de Lima, and Victoria. In addition, there are pools of water on the MA-21, near the Churriana intersection, and the intersections of Avenida Herrera Oria-Virgen de las Flores, Pasillo del Matadero Puente del Carmen, Pasillo Santa Isabel - Puente de la Aurora and Avenida Lope de Vega - Atabal have also been affected."
text2 = "There have also been some traffic jams and road closures in the capital, such as on Lope de Vega Avenue, heading towards the city center, near Julio Cortázar Avenue due to a road subsidence. Also there were some train delays in Madrid and a power blackout in Valencia. As the City Council indicated on social media, traffic has been diverted towards Colonia Santa Inés. There is also heavy traffic on the access roads from Juan XXIII Avenue to Plaza Manuel Azaña, Blas Infante Avenue, Guerrero Strachan Avenue, Velázquez Avenue, Camino Suárez, the Azucarera - Interhorce road, Santa Rosa de Lima, and Victoria. In addition, there are pools of water on the MA-21, near the Churriana intersection, and the intersections of Avenida Herrera Oria-Virgen de las Flores, Pasillo del Matadero Puente del Carmen, Pasillo Santa Isabel - Puente de la Aurora and Avenida Lope de Vega - Atabal have also been affected."

In [ ]:
df_resp

####  TODO fix hanging of output
“Please return one complete answer and then stop.” 

In [ ]:
client.responses.parse?

#### memory and model shredding 

In [ ]:
# !hf cache scan

## Run

In [ ]:
# %%scalene --reduced-profile  
# # # Turn profiling on
# scalene_profiler.start()



# Settings
# model_name = "meta-llama/Llama-3.1-8B-Instruct"
# model_name = "meta-llama/Meta-Llama-3-70B-Instruct"  # [144 GB VRAM]
model_name = "openai/gpt-oss-20b"  # bit bette performance than llama-3-70B but much smaller [43 GB VRAM]


# #gguf_filename = "models--unsloth--Llama-4-Maverick-17B-128E-Instruct-GGUF"#"Llama-4-Maverick-17B-128E-Instruct-UD-Q4_K_XL.gguf"
# model_name = "unsloth/Llama-4-Maverick-17B-128E-Instruct-GGUF"
# gguf_filename = "Llama-4-Maverick-17B-128E-Instruct-UD-Q4_K_XL.gguf"


time0 = time.time()

# clean up before applying CUDA
gc.collect()
torch.cuda.empty_cache() 
print(torch.cuda.memory_reserved() / 1e9)
torch.no_grad()

print(os.environ["CUDA_VISIBLE_DEVICES"])
# os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
print(os.environ["CUDA_VISIBLE_DEVICES"])


# Questions
question_1 = "Which infrastructure failures are mentioned in the text? Categorize the output by the type of infrastructure, the location, its location type (e.g., region, county, city, river etc.), the type of damage and potentially quantitative information about the damage."
question_2 = "Is the location of each affected or damaged critical infrastructure correctly identified?"


# ## init LLM extraction models
# decoder_model_1 = OpenAIModel(
#     model_name,
#     system_prompt=em.load_prompt_template(template_filename="gpt_system_prompt.txt")
# )
# decoder_model_2 = OpenAIModel(
#     model_name,
#     system_prompt=em.load_prompt_template(template_filename="gpt_system_prompt_step2.txt")
# )

# # decoder_model_1 = em.DecoderModelCaching(
# #     model_name,
# #     static_prompt=em.load_prompt_template(template_filename="short_static_llama3_NER.txt",)
# # )

# # decoder_model_2 = em.DecoderModelCaching(
# #     model_name,
# #     static_prompt=em.load_prompt_template(template_filename="short_static_llama3_NER_geollm_step2.txt",)
# # )


# load tokenizer
embed_model =  "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained(embed_model),
    max_tokens=256, # max tokens for MiniLM-l6-v2, set here explicitly
    # standardize input sizes of chunks for Llama models
    padding=True, # add zero as extra tokens to too short sequences so that they have the same length as other chunks
    truncation=True, # truncates too long sequences (> max_tokens). If False, they will be split into multiple chunks
)

## init chunker - based on hierachical chunker but also considers max token leng, merge smaller chunks, except when at end of paragraph (merge_peers=True)
chunker = HybridChunker(
    tokenizer=tokenizer,
        # max_tokens=256, # max tokens for MiniLM-l6-v2, set here explicitly
        # chunk_overlap=0, # no overlap between chunks, as we use merge_peers to merge smaller chunks and avoid splits in sentences
    split_by_sentence=True, # split by sentence first before merging smaller chunks, to avoid splits in sentence middle
    merge_peers=True,  # optional, defaults to True
)



gc.collect()
torch.cuda.empty_cache() 
torch.no_grad()
print(torch.cuda.memory_reserved() / 1e9)

# print(os.environ["CUDA_VISIBLE_DEVICES"])
# os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
# print(os.environ["CUDA_VISIBLE_DEVICES"])

# for Doc parsing and cleaning
md_converter = DocumentConverter(allowed_formats=[InputFormat.MD])


# CI-GEO pairs
geolocs_cache = geonamescache.GeonamesCache()
countries = geolocs_cache.get_countries()
ci_geo_countries = [*u.gen_dict_extract(countries, 'name')]



## init outputs
df_resp_step1_all = pd.DataFrame()
df_resp_step2_all = pd.DataFrame()
responses_error_list = [] 
df_ci_cases_not_grouped = pd.DataFrame()
df_geollama_response = pd.DataFrame()


if test_mode:
    search_path = docs_list_sample
    print("Test mode is ON. Using only a small sample of documents for testing.")
else:
    search_path = glob(str(Path(PARSED_TEXT_DIR, "*cleaned.jsonl")))




## Start CI impact extraction
for file_no, filename in enumerate(search_path):

    EXPORT_TYPE = ExportType.DOC_CHUNKS
    mislabeled: List[DocItem] = []
    min_paragraph_size = 50  
    temp_docs: List[ByteStream] = []
    temp_meta: List[Dict[str, str]] = []
    i: int
    combined_paragraph: str = ""
    combined_chars: int = 0
    para_num: int = 0
    section_name: str = ""
    page_no: Optional[int] = None
    first_note: bool = False


    time1 = time.time()

    src_language_nonengl = None 

    no_documents = len(search_path)
    filepath = Path(filename)
    filename_stem = filepath.stem


    print(f"\n\n ######## -------- Processing document [{file_no+1}/{no_documents}]: {filepath.name} -------- ######## \n")

    ## extract authors, publication year and title 
    author, year, title = dc.extract_citation_info(filename_stem)
    citation = f"{author} {year}".replace("  ", " ").strip()
    title = title.replace(" - ", "").replace("_cleaned", "").strip()



    # init dfs to store interim results for each doc
    df_resp_step1 = pd.DataFrame(
        columns=[
            "citation_id",
            "chunk_id",
            "infrastructure_type",
            "damage",
            "damage_value",
            "location",
            "location_type",
            "ci_entity",
            "geo_entity",
            "chunk_text"
        ]
    )
    df_resp_step2 = pd.DataFrame(
        columns=[
            "citation_id",
            "chunk_id",
            "infrastructure_type",
            "infrastructure_group",
            "damage",
            "damage_value",
            "location",
            "location_type",
            "ci_entity",
            "geo_entity",
            "coord_potential_locations",
            "chunk_text"
        ]
    )
    
    print(f"\n ##### ------- Cleaning document -----------########")

    pdf_filepath = os.path.join(DOCS_DIR, Path(filename_stem + ".pdf"))
    md_filename = filename_stem + ".md"
    md_filepath = os.path.join(PARSED_TEXT_DIR, Path(md_filename))
    cleaned_md_filepath = md_filepath.replace(".md", "_cleaned.md")
    # cleaned_jsonl_filepath = md_filepath.replace(".md", "_cleaned.jsonl")

    time_cleaning = time.time()

    if os.path.exists(cleaned_md_filepath):
        print(f"Cleaned markdown file already exists, loading file and proceeding with Ci impact extraction")
        # Load the existing cleaned markdown file
        doclingdoc = md_converter.convert(cleaned_md_filepath).document      

    elif os.path.exists(pdf_filepath):
        print(f"Cleaned markdown file does not exist yet. Cleaning document by using original PDF: '{filename}'")

        # get language of document
        src_language_doc = langdetect.detect(filename_stem.lower())  # lower case improves language detection


        ## Document converter with OCR
        print("Using OCR for text extraction as it identifies section titles, footers/headers and pagenumbers as such, but reads in also figure text sometimes") 
        # NOTE all other standard doclingConverter retunr section/headers etc as BODY not FURNITURE
        # NOTE: partly reads in figure text and table text 
        pdf_doc_org = dc.DocumentParser().ocr_converter.convert(source=pdf_filepath).document

        ##  get only list of Doc.items
        texts = dc.get_processed_texts(pdf_doc_org) 

        texts_clean = []
        section_names = []

        # text cleaning, annotating section names, remove header/footers, and translation
        for i, text in enumerate(texts):
            

    ### TODO make as Parser Doc class
            # get next text only when it is not page header/footer
            next_text = dc.get_next_text(texts, i)
            # page_no = get_current_page(text, combined_paragraph, page_no)


            # Update section header if the element is a section header
            # TODO: Need a stronger check on section headers that takes top of page into account, etc
            if dc.is_section_header(text) and text not in mislabeled:
                print("!!  Section header found:", text.text)
                section_name = text.text
                continue

            if dc.is_reference_section(section_name):
                print("Reference section found. Stopping further processing of document.")
                break  

            if dc.should_skip_element(text):
                continue
            
            # clean from double whitespace, newlines, etc.
            p_str = dc.clean_text(text.text)

            # clean from potential figure references
            p_str = dc.remove_figure_references(p_str)

            ## replace e.g. and i.e. --> eg and ie to avoid sentence splits
            p_str = re.sub(r"e\.g\.\s+", "eg ", p_str)
            p_str = re.sub(r"i\.e\.\s+", "ie ", p_str)

            # Removing URLs 
            # LangExtract tries to open these URLs when they occur in the document text
            # p_str= re.sub(r"http\S+", "", p_str) 

            p_str_chars = len(p_str)

            # If the paragraph does not end with final punctuation, accumulate it
            if not dc.is_sentence_end(p_str):
                combined_paragraph = dc.combine_paragraphs(combined_paragraph, p_str)
                combined_chars += p_str_chars
                continue

            # p_str ends with a sentence end; decide whether to process or accumulate it
            total_chars = combined_chars + p_str_chars
            if dc.is_section_header(next_text):
                # Immediately process if the next text is a section header
                p_str = dc.combine_paragraphs(combined_paragraph, p_str)
                combined_paragraph, combined_chars = "", 0
            elif total_chars < min_paragraph_size:
                # Not enough characters accumulated yet; decide based on next_text
                if next_text is None or (not dc.is_page_text(next_text) and dc.is_sentence_end(p_str)):
                    # End of document or next text item is not a text item and current paragraph ends with punctuation
                    # Process the paragraph and reset the accumulator even though this is a short paragraph
                    p_str = dc.combine_paragraphs(combined_paragraph, p_str)
                    combined_paragraph, combined_chars = "", 0
                else:
                    # Combine with next paragraph
                    combined_paragraph = dc.combine_paragraphs(combined_paragraph, p_str)
                    combined_chars = total_chars
                    continue
            else:
                # Sufficient characters: process the paragraph and reset the accumulator
                p_str = dc.combine_paragraphs(combined_paragraph, p_str)
                combined_paragraph, combined_chars = "", 0

            p_str = dc.combine_hyphenated_words(p_str)
            if p_str:  # Only add non-empty content
                para_num += 1
                dc.add_paragraph(
                    p_str, 
                    #para_num, section_name, page_no, 
                    temp_docs, 
                    # temp_meta
                )
                page_no = None
            
            print("\n-__ Paragraph #", para_num, "; Section:", section_name)

            # print(p_str)
            # print(temp_docs, temp_meta)

### TODO translator class
            ## Translation
            if src_language_doc != "en":

                supported_languages = ["fr", "de", "es", "it", "nl"]
                # supported_languages = ["en", "fr", "de", "es", "pt", "it", "pl", "cs", "nl", "da", "sv", "no", "hr", "ro", "bg", "sl", "sk", "lt", "et" ],
                if src_language_doc not in supported_languages:
                    print(f"Unsupported source language: {src_language_doc}. Continue with extraction on original text")
                    continue 

                print(f"Translating {src_language_doc} --> en")
                try:
                    p_str = tm.translate_2_english(src_language_doc, p_str)     
                except Exception as e:
                    print(f"! Cannot translate text, going to next chunk: {p_str}")
                    continue

            # collect cleaned text + meta data per doc            
            texts_clean.append(p_str)
            section_names.append(section_name)


        print("TEST Saving cleaned text as DoclingDocument to store text (p_str)+ meta (section title, page_no): ")
        # Initialize new DoclingDoc
        doclingdoc = DoclingDocument(schema_name="DoclingDocument",  version="1.0.0",name="My Custom Document")
        
        # write text and structural elements (titles, sections, and paragraphs) to doc     
        title_node = doclingdoc.add_title(text=filename_stem)
        for text, section_name in zip(texts_clean, section_names):
            print(section_name, ":", text)
            section_node = doclingdoc.add_heading(text=section_name, level=1, parent=title_node)
            doclingdoc.add_text(label=DocItemLabel.TEXT, text=text, parent=section_node)

        # Save doclingDocument as MD
        doclingdoc.save_as_markdown(cleaned_md_filepath) 

    # no pdf or md exists, skipping document
    else:    
        print(f"Neither cleaned markdown file nor original pdf file exists for document: '{filename}'. Skipping document.")
        continue

    print(f"Document cleaning took, {np.round((time.time() - time_cleaning) / 60, 1)} minutes")


    print("Chunking document...")
    chunk_iter = chunker.chunk(dl_doc=doclingdoc)  # NOTE cannot use chunker when re-created Docl.Document with cleaned text and old DoclingObject (from converter)
    doc = list(chunk_iter)
    # print(len(chunks), "chunks created with chunker.chunk(dl_doc=doclingdoc)")
    # print(len(texts_clean), "cleaned text items in doclingdoc.texts")
    
    # TODO 
    # test if contextualization improves model performance 

    # # apply contextualization (add section_name etc to chunk textfor better LLM unterstanding)
    # for i in range(len(chunks)):
    #     print(f"\nChunk {i} content before contextualization:\n{chunks[i].text}")
    
    #     ser_text = chunker.contextualize(chunk=chunks[i])
    #     ser_tokens = tokenizer.count_tokens(ser_text)
    #     print(f"chunker.contextualize(chunk) ({ser_tokens} tokens):\n{ser_text!r}")
    #     doc = ser_text
    #     print()
    


In [ ]:
# he scalene extension is already loaded. To reload it, use:
#   %reload_ext scalene
# ERROR: Do not try to invoke `start` if you have not called Scalene using one of the methods
# in https://github.com/plasma-umass/scalene#using-scalene
# # (The most likely issue is that you need to run your code with `scalene`, not `python`).

## TODO 


In [ ]:
resp.rpartition('"')[-3] + "]}"

In [ ]:
# print(f"Safety: saving responses (Step 1 + 2) for doc: {citation} ")
# df_resp_step1.to_csv(f"./interim_results/llm1_geollm_step1_{citation}.csv", encoding='utf-8', index=False)
# df_resp_step2.to_csv(f"llm1_geollm_step2_{citation}.csv", encoding='utf-8', index=False)

print(df_resp_step2.chunk_text[0])



## Doc. cleaning improve

In [ ]:
# c = """ 
# blublub title\n\n\n

# HERE IS NEW SUBSECTION:\n

# (large-scale) societal dis- ruptions dis - ruptions (Garschagen and Sandholz, 2018; Hallegatte et al., 2019; Fekete and Sandholz, 2021), empirical evidence on the impacts of extreme weather events on these systems is still

# Published by Copernicus Publications on behalf of the European Geosciences Union.

# E. E. Koks et al.: Flood impacts to infrastructure

# limited. This brief communication provides an overview of the observed ﬂood impacts to large-scale infrastructure sys- tems during the 2021 mid-July western European ﬂood event and how reconstruction of these large-scale systems has pro- 




# HERE IS NEW SUBSECTION

# severely damaged railway line (between the vil- lages of Spa and Pepinster) was reopened again on 3 Octo- ber 2021 (Rozendaal, 2021b). In the Netherlands, no large- scale damage has been reported to transport infrastructure. A few national highways were partly ﬂooded (e.g. the A76 in both directions) or brieﬂy closed (&lt; 3 d) because of the po- tential of ﬂooding. Most likely due to relative low-ﬂow ve- locities, damage to Dutch national road infrastructure was limited. Several railway sections were closed (e.g. the rail-

# way section between Maastricht and Liége) and some dam- age occurred to the railway infrastructure, in particular to the electronic “track circuit” devices and saturated railway em- bankments (Prorail, 2021).

# """
# #c = c.replace(r"\n", r" ")   # Isssue replaces also multipelinebreas eg before subsection
# c = re.sub(r"([^\s-])\n([^\s-])", r"\1 \2", c) # replace linebreak symbols when they occur just once, with whitespace (two linebreaks - probably new subsection)
# c = c.replace("/\n{2,}/g", "\n")  # remove linebreaks only when they occurred just once, but not for multiple linebreaks (e.g. before subsection)
# # Matches \n not preceded or followed by \n
# # c = re.sub(r"(?<!\n)\n(?!\n)", r"\n", c)  # remove linebreaks only when the yoccured just once, but not for multiple linebreaks (e.g. before subsection)
# c = re.sub(r"\s+", " ", c)  # replace >1 whitespaces with single whitespace

# # c = c.replace(r"\w*- ", "\w*-", c)  # removes any word followed by "-"
# # c = re.sub(r"([^\s-])-\n([^\s-])", r"\1\2", c)  # remove hypen and linebreaks TODO test with koks sentences
# c = re.sub(r"([^\s-])- ([^\s-])", r"\1-\2", c)  # remove hypens in the middle of lines

# c 


# # 0090- some weird breaks-
# # And some long sentences 
# # which are not separated by dots but by line breaks and hyphens 



In [ ]:
# !uv pip install "unstructured[pdf]"  # unstructured  #langchain-unstructured #langchain-community
# # # !uv add langchain
# !uv lock
# !uv sync
# from langchain_community.document_loaders import DirectoryLoader, UnstructuredFileLoader
# #from langchain.loaders import , UnstructuredFileLoader

# loader = DirectoryLoader(str(Path(DOCS_DIR)),  loader_cls=UnstructuredFileLoader, show_progress=True)
# pdf_docs = loader.load()
# # glob=glob("*.pdf"),
# print(f"Number of Documents: {len(pdf_docs)}")

# # convert the different layouts of the pdf files into unified markdown format incl. sub/section titles, tables, caption text etc
# for idx, doc in enumerate(pdf_docs, start=0):
#     print(doc)

In [ ]:
# pdf_filepath = Path("Lloyd's List 2024 - Port of Valencia reopens after devastating floods.pdf")
# # Path(DOCS_DIR, "Lloyd's List 2024 - Port of Valencia reopens after devastating floods.pdf")
# print(pdf_filepath)
# print("Remove reference section")

# # setup converter for PDF and markdown
# converter = DocumentConverter(
#     allowed_formats=[InputFormat.PDF, InputFormat.MD],
#     format_options={
#         InputFormat.PDF: FormatOption(
#             pipeline_cls=StandardPdfPipeline,
#             pipeline_options=pipeline_options,
#             backend=PyPdfiumDocumentBackend,
#         ),
#     },
# )
# pdf_text = converter.convert(pdf_filepath).document
    
# # loader = DoclingLoader("/beegfs/scratch/a-buch/_PROJECTS/data/text_sources/Lloyd's List 2024 - Port of Valencia reopens after devastating floods.pdf")  # use chunks from Docling.Loader
# # pdf_doc = loader.load()